# Sweeps: resolve and declare

A small example using two sources and two tokenizer sizes. Resolution builds a concrete dependency graph; declaration writes its manifests. These cells do not download data or run jobs.

Run with the project's Python environment, from the repository root or `notebooks/`.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import lab
from artifacts.core.resolve import resolve
from artifacts.core.sweep import Sweep
from artifacts.sources import SourceURL
from artifacts.tokenized import TokenizedSource
from artifacts.tokenizers.bpe import Tokenizer


## 1. Resolve one artifact

`resolve()` always returns a list of concrete artifacts, with dependencies first.


In [ ]:
books = SourceURL(name="books", url="https://example.org/books.txt")
code_source = SourceURL(name="code", url="https://example.org/code.txt")

for artifact in resolve(books):
    print(artifact.artifact_path)
    

sources/books


artifacts.sources.SourceURL

## 2. Sweep a parameter and a dependency

The tokenizer has two vocabulary sizes. The tokenization artifact has two source choices. Together they produce **2 × 2 = 4 tokenization artifacts**.

The vocabulary sweep is inside a dependency, so this also demonstrates recursive expansion.


In [5]:
tokenizer = Tokenizer(
    vocab_size=Sweep((100, 200)),
    special_tokens=("<pad>",),
    sources=(books, code_source),
)

tokens = TokenizedSource(
    tokenizer=tokenizer,
    source=Sweep((books, code_source)),
)
resolve(tokens)

[SourceURL(name='books', url='https://example.org/books.txt'),
 SourceURL(name='code', url='https://example.org/code.txt'),
 Tokenizer(vocab_size=100, special_tokens=('<pad>',), sources=frozenset({SourceURL(name='code', url='https://example.org/code.txt'), SourceURL(name='books', url='https://example.org/books.txt')})),
 TokenizedSource(tokenizer=Tokenizer(vocab_size=100, special_tokens=('<pad>',), sources=frozenset({SourceURL(name='code', url='https://example.org/code.txt'), SourceURL(name='books', url='https://example.org/books.txt')})), source=SourceURL(name='books', url='https://example.org/books.txt')),
 Tokenizer(vocab_size=200, special_tokens=('<pad>',), sources=frozenset({SourceURL(name='code', url='https://example.org/code.txt'), SourceURL(name='books', url='https://example.org/books.txt')})),
 TokenizedSource(tokenizer=Tokenizer(vocab_size=200, special_tokens=('<pad>',), sources=frozenset({SourceURL(name='code', url='https://example.org/code.txt'), SourceURL(name='books', url

The full graph contains two sources, two tokenizers, and four tokenization artifacts. Shared dependencies appear once.


In [6]:
for artifact in artifacts:
    print(type(artifact).__name__, artifact.artifact_path)

# Asking for the same declaration twice still produces the same eight paths.
assert resolve(Sweep((tokens, tokens))) == artifacts


NameError: name 'artifacts' is not defined

## 3. Preview a declaration

Use a temporary local folder. The default `commit=False` reports what would be declared and writes nothing.


In [18]:
scratch = TemporaryDirectory(prefix="launchpad-sweeps-")
root = Path(scratch.name)

preview = lab.declare(tokens, root=root)

assert len(preview.root_paths) == 4
assert len(preview.rows) == 8
assert list(root.iterdir()) == []


sources/books                       new
sources/code                        new
tokenizers/bpe-100-a026bc9b50       new
tokenized/bpe-100-a026bc9b50/books  new
tokenizers/bpe-200-02a89bed1d       new
tokenized/bpe-200-02a89bed1d/books  new
tokenized/bpe-100-a026bc9b50/code   new
tokenized/bpe-200-02a89bed1d/code   new

8 new
ok -- 8 to declare


## 4. Write the manifests

`commit=True` publishes each unique artifact's manifest. This prepares the jobs; it does not execute them.


In [19]:
# Empty notebook bytes skip notebook auto-discovery for this local example.
report = lab.declare(tokens, root=root, commit=True, notebook=b"")

manifests = sorted(root.rglob("manifest.json"))
assert len(manifests) == 8
print(f"\nWrote {len(manifests)} manifests under {root}")


sources/books                       declared  (created)
sources/code                        declared  (created)
tokenizers/bpe-100-a026bc9b50       declared  (created)
tokenized/bpe-100-a026bc9b50/books  declared  (created)
tokenizers/bpe-200-02a89bed1d       declared  (created)
tokenized/bpe-200-02a89bed1d/books  declared  (created)
tokenized/bpe-100-a026bc9b50/code   declared  (created)
tokenized/bpe-200-02a89bed1d/code   declared  (created)

8 declared
8 manifests written: sources/books, sources/code, tokenizers/bpe-100-a026bc9b50, tokenized/bpe-100-a026bc9b50/books, tokenizers/bpe-200-02a89bed1d, tokenized/bpe-200-02a89bed1d/books, tokenized/bpe-100-a026bc9b50/code, tokenized/bpe-200-02a89bed1d/code

Wrote 8 manifests under /var/folders/_9/c3pgt8bn23n2jqgj_1cfghrm0000gn/T/launchpad-sweeps-ab3qlhde


## 5. Declare it again

The existing manifests are reused. `root_paths` identifies the four requested outputs; report rows also include their dependencies.


In [20]:
again = lab.declare(tokens, root=root, commit=True, notebook=b"")
assert not any(row["created"] or row["updated"] for row in again.rows)

print("\nRequested outputs:")
for path in again.root_paths:
    print(path)
    

# When finished inspecting the local files, run: scratch.cleanup()


sources/books                       declared
sources/code                        declared
tokenizers/bpe-100-a026bc9b50       declared
tokenized/bpe-100-a026bc9b50/books  declared
tokenizers/bpe-200-02a89bed1d       declared
tokenized/bpe-200-02a89bed1d/books  declared
tokenized/bpe-100-a026bc9b50/code   declared
tokenized/bpe-200-02a89bed1d/code   declared

8 declared
ok -- 0 to declare

Requested outputs:
tokenized/bpe-100-a026bc9b50/books
tokenized/bpe-200-02a89bed1d/books
tokenized/bpe-100-a026bc9b50/code
tokenized/bpe-200-02a89bed1d/code
